# Low-resolution double-basin PG prototype

This production-oriented notebook solves an idealized version of the attached two-basin sketch. Two deep bowls are separated by a shallow central ridge that extends westward from the eastern shelf, leaving a deeper western gateway analogous to Fram Strait. The outer shallow region represents a continental shelf.

The calculation uses a constant Coriolis parameter (an $f$ plane) and cyclonic wind-stress cells over both basins. Expensive adjoint, perturbation, and external-theory checks from `PG_API_Example.ipynb` are omitted. The depth-integrated momentum and three equivalent vorticity-budget closure diagnostics are retained.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PlanetaryGeostrophy"))
using PlanetaryGeostrophy
using Plots
using Test

## Fast prototype controls

Increase `mesh_resolution`, `horizontal_points`, `vertical_points`, and `diagnostic_points` only after the geometry and forcing have been finalized. Disk output is disabled by default.

In [ ]:
mesh_resolution = 0.16
horizontal_points = 33
vertical_points = 25
diagnostic_points = 17
SAVE_OUTPUT = false

output_path(name) = SAVE_OUTPUT ? joinpath(@__DIR__, name) : nothing

## Idealized double-basin geometry and forcing

The bathymetry is a shallow shelf plus northern and southern Gaussian deep basins. A smaller Gaussian deepening on the western side cuts a gateway through the otherwise shallow saddle between them. All depth derivatives are supplied analytically. The two compact wind-stress cells have positive (cyclonic) curl near their basin centers.

In [ ]:
Lx, Ly = 2.0, 2.8
ax, ay = Lx / 2, Ly / 2
boundary = ellipse_boundary(ax, ay; points=64)
parameters = PGParameters(0.02, 0.20; viscosity=1.0, diffusivity=1.0)

gaussian(x, y, cx, cy, sx, sy) = exp(-((x - cx) / sx)^2 - ((y - cy) / sy)^2)
gaussian_x(x, y, cx, cy, sx, sy) =
    -2(x - cx) / sx^2 * gaussian(x, y, cx, cy, sx, sy)
gaussian_y(x, y, cx, cy, sx, sy) =
    -2(y - cy) / sy^2 * gaussian(x, y, cx, cy, sx, sy)

shelf_depth = 0.30
north_basin = (amplitude=0.95, cx=0.05, cy=0.67, sx=0.62, sy=0.58)
south_basin = (amplitude=0.85, cx=0.00, cy=-0.70, sx=0.66, sy=0.55)
western_gateway = (amplitude=0.38, cx=-0.67, cy=0.00, sx=0.25, sy=0.42)
features = (north_basin, south_basin, western_gateway)

depth(x, y) = shelf_depth + sum(
    feature.amplitude * gaussian(x, y, feature.cx, feature.cy, feature.sx, feature.sy)
    for feature in features
)
depth_x(x, y) = sum(
    feature.amplitude * gaussian_x(x, y, feature.cx, feature.cy, feature.sx, feature.sy)
    for feature in features
)
depth_y(x, y) = sum(
    feature.amplitude * gaussian_y(x, y, feature.cx, feature.cy, feature.sx, feature.sy)
    for feature in features
)

f0 = 1.0
stress_amplitude = 1e-3
wind_cells = (
    (cx=north_basin.cx, cy=north_basin.cy, sx=0.55, sy=0.52),
    (cx=south_basin.cx, cy=south_basin.cy, sx=0.57, sy=0.50),
)
stress_envelope(x, y, cell) = gaussian(x, y, cell.cx, cell.cy, cell.sx, cell.sy)
wind_stress(x, y) = sum(
    stress_amplitude * stress_envelope(x, y, cell) *
    (-(y - cell.cy) / cell.sy + im * (x - cell.cx) / cell.sx)
    for cell in wind_cells
)
wind_stress_curl(x, y) = sum(
    stress_amplitude * stress_envelope(x, y, cell) * (
        (1 - 2((x - cell.cx) / cell.sx)^2) / cell.sx +
        (1 - 2((y - cell.cy) / cell.sy)^2) / cell.sy
    ) for cell in wind_cells
)

problem = PGProblem(
    parameters=parameters,
    depth=depth,
    depth_x=depth_x,
    depth_y=depth_y,
    coriolis=(x, y) -> f0,
    coriolis_x=(x, y) -> 0.0,
    coriolis_y=(x, y) -> 0.0,
    wind_stress=wind_stress,
    wind_stress_curl=wind_stress_curl,
    restoring_buoyancy=(x, y, z) -> 1e-2z,
    surface_buoyancy=(x, y) -> 2e-3 * y / Ly,
    boundary=boundary,
)

In [ ]:
geometry_x = range(-0.99ax, 0.99ax; length=101)
geometry_y = range(-0.99ay, 0.99ay; length=141)
geometry_depth = [
    problem.inside(xx, yy) ? depth(xx, yy) : NaN
    for yy in geometry_y, xx in geometry_x
]
geometry_figure = contourf(
    geometry_x, geometry_y, geometry_depth;
    levels=14, color=:deep, aspect_ratio=:equal,
    xlabel="x", ylabel="y", title="Double-basin depth H(x, y)",
)
display(geometry_figure)

## Solve and sample

The pressure mesh and three-dimensional sampling grids are deliberately coarse for rapid iteration.

In [ ]:
solution = solve_pg(
    problem;
    mesh_file=joinpath(@__DIR__, "double_basin_prototype_pressure.msh"),
    mesh_resolution=mesh_resolution,
    overwrite_mesh=true,
)

x = range(-0.98ax, 0.98ax; length=horizontal_points)
y = range(-0.98ay, 0.98ay; length=horizontal_points)
maximum_depth = maximum(depth(xx, yy) for yy in y, xx in x if problem.inside(xx, yy))
z = range(-maximum_depth, 0.0; length=vertical_points)
fields = sample_solution(solution, x, y, z)

solution_figure = plot_solution(
    fields;
    sigma_level=0.5,
    output=output_path("double_basin_solution.png"),
)
display(solution_figure)

if SAVE_OUTPUT
    display(write_solution(fields, joinpath(@__DIR__, "double_basin")))
end

## Shared diagnostic fields

All budget calculations reuse this modest diagnostic grid and the resulting stress and transport samples.

In [ ]:
diagnostic_x = range(-0.94ax, 0.94ax; length=diagnostic_points)
diagnostic_y = range(-0.94ay, 0.94ay; length=diagnostic_points)
diagnostics = compute_diagnostics(solution, diagnostic_x, diagnostic_y)
display(performance_summary(diagnostics))

display(plot_depth_integrated_current(
    diagnostics; arrow_scale=0.70,
    output=output_path("double_basin_transport.png"),
))
display(plot_streamfunction(
    diagnostics;
    output=output_path("double_basin_streamfunction.png"),
))

## Budget closure diagnostics

Retain the depth-integrated momentum balance and the barotropic, depth-integrated, geostrophic-contour, and transport-divergence vorticity balances from the API example.

In [ ]:
dim = compute_dim_diagnostics(solution, diagnostics)
barotropic_vorticity = compute_barotropic_vorticity_diagnostics(solution, diagnostics)
vorticity = compute_vorticity_diagnostics(solution, diagnostics)
gcvb = compute_geostrophic_contour_vorticity_diagnostics(solution, diagnostics)
tdvb = compute_transport_divergence_vorticity_diagnostics(solution, diagnostics)

closure = (
    momentum_x=dim.timings.x_relative_l2_residual,
    momentum_y=dim.timings.y_relative_l2_residual,
    barotropic_vorticity=barotropic_vorticity.timings.relative_l2_residual,
    depth_integrated_vorticity=vorticity.timings.relative_l2_residual,
    geostrophic_contour_vorticity=gcvb.timings.relative_l2_residual,
    transport_divergence_vorticity=tdvb.timings.relative_l2_residual,
)
display(closure)

In [ ]:
display(plot_dim_balance(
    dim; output=output_path("double_basin_dim_balance.png"),
))
display(plot_barotropic_vorticity_balance(
    barotropic_vorticity;
    output=output_path("double_basin_barotropic_vorticity_balance.png"),
))
display(plot_vorticity_balance(
    vorticity; output=output_path("double_basin_vorticity_balance.png"),
))
display(plot_geostrophic_contour_vorticity_balance(
    gcvb;
    output=output_path("double_basin_geostrophic_contour_vorticity_balance.png"),
))
display(plot_transport_divergence_vorticity_balance(
    tdvb;
    output=output_path("double_basin_transport_divergence_vorticity_balance.png"),
))

## Automated closure checks

These smoke tests catch failed or empty diagnostics without imposing a final production-resolution tolerance. Tight closure thresholds should be selected after the mesh and sampling resolution are increased.

In [ ]:
@testset "PG budget closure smoke tests" begin
    @test count(diagnostics.wet) > 0
    @test count(barotropic_vorticity.wet) > 0
    @test count(vorticity.wet) > 0
    @test count(gcvb.wet) > 0
    @test count(tdvb.wet) > 0
    @test all(isfinite, values(closure))
end